# Round 3, RUN A — the same data, a LONGER stage 2

⚠ **This is still ROUND 3** (owner, 2026-09-01). The exam was already read once this round, on
`r3-final-stage2-last`; selecting among these runs is `_realval_v2`'s job, and a second exam read
is a rule question for the owner, not something this notebook assumes.

**One variable: stage 2 runs 4,000 steps instead of 2,000. Nothing else moves.** Same corpus
(`strips_v7_final`), same real pool (`strips_b8`, 3,929 strips), same `:5`, same lr, same mix.
Run B (`round3_runb_oldstrips_colab.ipynb`) adds the 1,435 retired-crop strips and is the *other*
variable — run them in two tabs, and the pair is what makes each one attributable.

## ⛔ Stage 1 is NOT re-run — it reuses Round 3's

Stage 1 is identical work to the Round-3 final run, and that checkpoint is on Drive. Re-running it
would cost ~2.5 h **and** break the comparison: training is not bit-deterministic, so a fresh
stage 1 would make Run A differ from Round 3 in two places instead of one. Total GPU here is the
stage-2 run alone, roughly **1.5–1.8 h**.

## Why 4,000 and why this is worth a run

Round 3's stage 2 real val loss fell **0.0301 → 0.0182 (−39%) and was still falling at the last
step**. ⚠ But that is NOT proof more steps help: the cosine LR reaches **zero** at `--max-steps`, so
the curve flattens at the end whether or not the model has converged. A 4,000-step run is a
*different schedule*, not an extension of the old one — which is exactly why it needs its own run
rather than an argument.

## ⭐ Read the `best-real` checkpoint — it is new, and it is why this run is readable

`train.py` now saves **three** checkpoints:

| tag | selected on |
|---|---|
| `best` | the blended val loss — **~92% synthetic by strip count** |
| **`best-real`** | **the REAL val loss alone** ⭐ |
| `last` | the final step (resume point) |

⛔ On Round 3's run the blend stamped `best` at **step 500** while real val kept improving to step
1750 — and that step-1750 model needed **22% fewer corrections** on `_realval_v2`. Over 4,000 steps
that gap gets worse, and without `best-real` the good checkpoint would sit between two evals and be
lost. ⚠ `best` is selected **exactly as before**, so this run stays comparable with Round 3.

## After the run

Bring **all three** home and choose between them on `_realval_v2` with `paired_arm_score.py`.
⛔ Never choose on the val losses printed here — real-val selects, the exam grades, once.


In [ ]:
# ===== THE ONLY KNOBS IN THIS NOTEBOOK — and the mix is NOT one of them =====
ARM = 'r3a'
STRIPS = 'data/synthetic/strips_v7_final'   # the 3-flag render: staccato + concave tuplet + usul barline
ZIP = 'tnc_round3_final_colab.zip'   # unchanged — Run A adds no data
DRIVE = '/content/drive/MyDrive/tnc'
# THE REAL POOL. One pool, current crops. ⛔ Never add strips_nota / strips_r1 / strips_tup.
REAL = 'data/real/rung3/strips_b8'
REPEAT = 5          # -> real ~33% of stage-2 batches at 3,539 train-side strips. See the header.
print(ARM, STRIPS, ZIP, f'| real: {REAL}:{REPEAT} | mix: train.py defaults (0.65/0.35, scan off)')

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy the package Drive -> VM disk and unzip (fast local disk for the dataloader).
# ⛔ CHECK THE COPY BEFORE TRUSTING IT. The Drive FUSE mount drops — "Transport endpoint is not
# connected" — most often when Drive has not finished processing a fresh upload. `cp` then leaves a
# TRUNCATED file, `unzip` fails, and the first error you actually see is a missing render_config.json
# three steps later, which looks like a bad zip and is not. Fail here instead, with the real reason.
import os, subprocess
ZIP_BYTES = 807713909   # `stat -f %z data/colab/tnc_round3_final_colab.zip` on the Mac

src = f'{DRIVE}/{ZIP}'
assert os.path.exists(src), (
    f'{src} not on Drive. Either the upload is unfinished or the mount is dead — '
    'Runtime > Restart session, re-run the mount cell, then `!ls -l {DRIVE}` to look.')
have = os.path.getsize(src)
assert have == ZIP_BYTES, (
    f'Drive has {have:,} bytes, expected {ZIP_BYTES:,}. The upload is INCOMPLETE or still syncing — '
    'wait for it to settle and re-run this cell. Do not unzip a partial file.')
print(f'Drive copy looks whole: {have:,} bytes')

# rsync, not cp: it is resumable, so a mount hiccup costs a retry rather than the whole transfer.
!rsync --progress {DRIVE}/{ZIP} /content/
got = os.path.getsize(f'/content/{ZIP}')
assert got == ZIP_BYTES, (
    f'copied {got:,} of {ZIP_BYTES:,} bytes — the mount dropped mid-copy. '
    'Restart the session and try again; re-run this cell, rsync will resume.')

# `unzip -t` reads the central directory, which is exactly what a truncated file lacks.
assert subprocess.run(['unzip', '-tq', f'/content/{ZIP}']).returncode == 0, \
    'the copied zip fails its own integrity test — delete /content/{ZIP} and re-copy'
print('zip integrity OK')

!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/{ZIP}

# WHICH CORPUS IS ACTUALLY ON DISK. All three flags are LABEL-FREE — they change pixels only, so the
# manifest cannot tell this corpus from an arm's. render_config.json is the ONLY place they are
# checkable, which is why they are asserted here as well as in make_round3_colab_zip.sh.
import json
cfg = json.load(open(f'/content/tnc/{STRIPS}/render_config.json'))
print(cfg)
assert cfg.get('staccatoNoise') is True,  'MISSING --staccato-noise — this is not the final render'
assert cfg.get('concaveTuplet') is True,  'MISSING --concave-tuplet — this is not the final render'
assert cfg.get('usulBarline') is True,    'MISSING --usul-barline — this is not the final render'
assert cfg['legacyTupletMark'] is False and cfg['thinSharps'] is True and cfg['printNoise'] is False
assert cfg.get('maxMeasures', None) is None

# WHICH REAL POOL IS ON DISK. The retired pools' filenames survive a re-slice and their pixels do
# not, so a name check is not enough on its own — but their ABSENCE is decisive.
import os
assert os.path.isdir(f'/content/tnc/{REAL}'), f'{REAL} missing from the zip'
for dead in ('strips_nota', 'strips_r1', 'strips_tup'):
    assert not os.path.isdir(f'/content/tnc/data/real/rung3/{dead}'), \
        f'RETIRED POOL {dead} IS IN THE ZIP — rebuild it with: sh scripts/make_round3_colab_zip.sh final'
!wc -l /content/tnc/{STRIPS}/manifest.jsonl
!wc -l /content/tnc/{REAL}/manifest.jsonl   # expect 3929
!python -c "import json;s=json.load(open('/content/tnc/data/split_v4.json'));print('train',len(s['train_pieces']),'val',len(s['val_pieces']))"

# ⛔ THE ZIP MUST CARRY THE PATCHED train.py. `best-real` (the checkpoint selected on the REAL val
# loss) was added 2026-09-01, and WITHOUT IT a 4,000-step run silently discards its best real-page
# checkpoint between evals — the exact failure Round 3 hit at 2,000 steps. A stale zip would train
# happily and give you the wrong model, so this fails loudly instead.
src = open('/content/tnc/src/vision/train.py').read()
assert 'best_real' in src and 'best-real' in src, (
    "STALE train.py IN THE ZIP — it has no `best-real` checkpoint.\n"
    "Fix: rebuild the zip on the Mac (`sh scripts/make_round3_colab_zip.sh final`) and re-upload,\n"
    "or drop the patched src/vision/train.py into MyDrive/tnc/ and copy it over this one.")
print('train.py: best-real checkpoint present ✅')

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# ===== THE FLAGS ARE THE RENDER — prove they are in the PIXELS before spending a GPU hour =====
# Nothing downstream records them: labels, manifest and split are what they would be with the flags
# off (188 strip labels over 4 scores are byte-identical with --usul-barline on and off). So the
# check is on the IMAGES, and on the mix being the default.
%cd /content/tnc
import sys
sys.path.insert(0, 'src/vision')
from augment import Augmenter
a = Augmenter(seed=7)
assert (a.photo_share, a.scan_share) == (0.35, 0.0), (a.photo_share, a.scan_share)
print(f'mix: screenshot {1-a.photo_share-a.scan_share:.2f} / photo {a.photo_share} / scan {a.scan_share}  (default)')

import json, random
from PIL import Image
rows = [json.loads(l) for l in open(f'{STRIPS}/manifest.jsonl')]
print(f'{len(rows)} synthetic strips')

# LOOK AT THEM. Each flag is coined PER PIECE, so a sample of one piece shows nothing — these draw
# from strips whose label gives the flag something to act on.
def show(pred, what, n=2):
    hits = [r for r in rows if pred(r)]
    print(f'{what}: {len(hits)} candidate strips')
    random.Random(7).shuffle(hits)
    for r in hits[:n]:
        print('  ', r['image'])
        display(Image.open(f"{STRIPS}/{r['image']}"))

show(lambda r: '.' in r['label'], 'staccato — dots ABOVE/BELOW noteheads, not beside')
show(lambda r: '\\tup3' in r['label'], 'tuplet — some pieces draw a CONTINUOUS arc with the 3 inside it')
show(lambda r: '|' in r['label'], 'usul barline — light DASHED rules INSIDE the bar, at the beat groups')

In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, ONE real pool listed, `exam-disjointness OK`,
# `augment=on (screenshot 0.65 / photo 0.35)`, and val loss FALLING.
# ⚠ READ THE `real pool ... : N train xR / M val strips` LINE — that is the only place the pool and
# its repeat are visible, and it is what the header's 33% rests on.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL} \
    --every-share 0.15 --out-dir /content/r3-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — before any long run =====
#   hours = (steps * batch) / samples_per_sec / 3600
# ⚠ Whatever you set, the STEP COUNTS AND BATCH SIZE must match the arms': 6000 @ 16, then 2000 @ 16.
# --num-workers and the GPU model do not change the result; those two do.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL} \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

In [ ]:
# ===== STAGE 1 — ⛔ DO NOT RUN THIS. REUSE ROUND 3's. =====
# Run A's ONE variable is the stage-2 step count, so stage 1 is IDENTICAL work to the Round-3 final
# run — same corpus, same split, same flags, same 6,000 steps — and that run's checkpoint is already
# on Drive at `r3-final-stage1/best`. Re-running it costs ~2.5 h of GPU.
# ⭐ AND REUSING IT IS THE MORE CORRECT CHOICE, not merely the cheaper one: training is not
# bit-deterministic on a GPU, so a re-run produces a slightly DIFFERENT stage-1 model, and Run A
# would then differ from Round 3 in two places instead of one. Sharing the stage-1 checkpoint is what
# makes "4,000 steps vs 2,000" the only variable.
STAGE1 = f'{DRIVE}/r3-final-stage1/best'
import os
assert os.path.isdir(STAGE1), (
    f'{STAGE1} missing. If Round 3 s stage 1 is genuinely gone, uncomment the cell below and spend '
    'the 2.5 h — but then say so when reporting, because A vs Round 3 is no longer a clean pair.')
print('reusing Round 3 stage 1:', STAGE1)

# --- only if the checkpoint above is really gone -------------------------------------------------
# %cd /content/tnc
# !python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
#     --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
#     --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 \
#     --eval-every 250 --save-every 250


In [ ]:
# ===== STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Fresh LOW lr + short warmup from the stage-1 checkpoint.
# ⚠ 4,000 STEPS — the one variable in Run A. Watch the `real` column every 250 steps: if it
# flattens well before the end, that answers the question without a further run.
# ⚠ `:5` is right for b8 ALONE (32.9% real). Run B adds strips_oldhuman and must drop to `:4`.
# ⚠ STARTS FROM ROUND 3's STAGE 1 (see the cell above) — that shared start is what keeps this a
# clean one-variable comparison against the Round-3 final model.
%cd /content/tnc
!python -u src/vision/train.py --model {STAGE1} \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL}:{REPEAT} \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 4000 --batch-size 16 --num-workers 10 \
    --eval-every 250 --save-every 250

In [ ]:
# RESUME after a disconnect: re-run the setup cells, then this with the SAME flags as the stage
# you were running (edit out-dir/flags to match). --resume reloads model+optimizer+scheduler from
# <out-dir>/last and ignores --model.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 \
    --eval-every 250 --save-every 250 --resume

In [ ]:
# ===== SANITY ONLY — did anything break? =====
# NOT the pre-registered number, and NOT the selection. Both are read on the Mac. This cell exists so
# a broken run is caught before it is downloaded, and to see `best` and `last` side by side.
%cd /content/tnc
!python src/vision/make_realval_pool.py --real-dir {REAL} --split data/split_v4.json

for ck in [f'r3-{ARM}-stage2/best', f'r3-{ARM}-stage2/best-real', f'r3-{ARM}-stage2/last']:
    print('=' * 70, '\n==', ck)
    !python src/vision/eval_omr.py --checkpoint {DRIVE}/{ck} \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0

## After the run

1. **Download ALL THREE** from `MyDrive/tnc/r3-r3a-stage2/` — `best`, **`best-real`**, `last`.
2. **Choose between them on `_realval_v2`** — real-val selects, which is legal:
   ```bash
   .venv-ml/bin/python scripts/rung3/paired_arm_score.py \
       --ctl data/checkpoints/r3-final-stage2-last --arm data/checkpoints/r3a-stage2-best-real \
       --pool data/real/rung3/_realval_v2
   ```
   ⚠ Compare against **`r3-final-stage2-last`** (Round 3's chosen model), not against `round2-stage2-best` —
   the question is whether the longer stage 2 bought anything over Round 3.
3. **Then compare Run A against Run B** the same way, before either goes near the exam.
4. ⛔ **The exam is one-shot per round.** It was read for Round 3 on 2026-09-01. Do not read it again
   until a Round-4 model is chosen.
